# TermpointToGNB Distancia (4G--5G)
### 1 - Utiliza o MO processado (5G_TermPointToGNB) para obter as informações de TermpointToGNB
### 2 - Utiliza o cellref 4G para obter informações de latitude e longitude da src
### 3 - Utiliza o cellref 5G para obter informações de latitude e longitude da tgt
### 4 - Expurgar sites DSS da tgt
### 5 - Criar a função distancia para calcular a distancia entre a src e a tgt 

In [ ]:
# Obtendo informações do TermpointToGNB

import pandas as pd

df= pd.read_csv('C:/Camilo/5G_TermPointToGNB.txt', sep='\t', encoding='ISO-8859-1')

df= df[df['Ancora'].str[:1]!='T']

df['Aux_LTE']= df['eNB'].str[1:4] + '_' +df['eNB'].str[-2:]
print(df.shape)
print('--'*30)
print(df.head())
print('--'*30)

In [ ]:
# Abrindo o cellref para inserir os dados de lat long da src

cellref= pd.read_excel('C:/Camilo/cellref.xlsx', usecols=['CLUSTER','SiteID', 'eNodeB_ID', 'Latitude', 'Longitude'])
print(cellref.shape)
cellref.drop_duplicates(inplace=True)
cellref['Aux_LTE']= cellref['SiteID'].str[1:4] + '_' +cellref['SiteID'].str[-2:]
print(cellref.shape)
print(cellref.head())

In [ ]:
# Juntando os dados de Termpoint com cellref LTE para obtermos o lat long da src

df1= pd.merge(df, cellref, on=['Aux_LTE'], how='left')

df1.rename(columns={'Latitude':'Latitude_src', 'Longitude':'Longitude_src'},inplace=True)
df1.drop(columns=['SiteID', 'eNodeB_ID'], inplace=True)
df1.drop_duplicates(subset=['UF', 'eNB', 'termPointToGNB_gNBId'], inplace=True)
print(df1.shape)
df1.head()

In [ ]:
# Abrindo o cellref 5G para inserir os dados de lat long da tgt

cellref_NR= pd.read_excel('C:/Camilo/cellref_5G.xlsx', usecols=['SiteID', 'GNodeBID', 'Latitude', 'Longitude'])
print(cellref_NR.shape)

cellref_NR.drop_duplicates(inplace=True)


print(cellref_NR.shape)
print(cellref_NR.head())

In [ ]:
# juntando as informações do df1 (TermpointToGNB com lat e long da src) com o cellref 5G para adicionarmos o lat long da tgt 

df2= pd.merge(df1, cellref_NR, left_on=['termPointToGNB_gNBId'], right_on=['GNodeBID'], how='left')
df2.drop_duplicates(subset=['UF', 'eNB', 'termPointToGNB_gNBId'], inplace=True)

df2.rename(columns={'Latitude':'Latitude_tgt', 'Longitude':'Longitude_tgt'},inplace=True)
df2.drop(columns=['Aux_LTE', 'SiteID', 'GNodeBID'], inplace=True)
df2.head()

In [ ]:
# Criando a função para calcular a distancia entre a src e a tgt

import pandas as pd
from haversine import haversine, Unit

def calculate_distance(row):
    # Verifica se algum dos valores de coordenadas é None
    if any(value is None for value in [row['Latitude_src'], row['Longitude_src'], row['Latitude_tgt'], row['Longitude_tgt']]):
        return None

    # Se todos os valores estiverem presentes, calcula a distância
    src_coords = (row['Latitude_src'], row['Longitude_src'])
    tgt_coords = (row['Latitude_tgt'], row['Longitude_tgt'])
    return haversine(src_coords, tgt_coords, unit=Unit.KILOMETERS)

# Usando apply para aplicar a função distancia no dataframe
df2['Distancia'] = df2.apply(calculate_distance, axis=1).round(2)


df2.head()

In [ ]:
# Criando a coluna Regional

df2['Regional']= df2['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

# Para alterar a posição das colunas. Primeiro devemos obter a lista atual de colunas.
colunas = df2.columns.tolist()

# Vamos criar uma nova lista sem 'Regional' e 'CLUSTER'
novas_colunas = [col for col in colunas if col not in ['Regional', 'CLUSTER']]

# Agora, insira 'REGION' e 'CLUSTER' nas posições desejadas
novas_colunas.insert(2, 'Regional')  # Insere 'REGION' na terceira posição
novas_colunas.insert(4, 'CLUSTER') # Insere 'CLUSTER' na quinta posição

# Reorganize as colunas do DataFrame
df2 = df2[novas_colunas]

df2.rename(columns={'CLUSTER': 'MUNICIPIO'}, inplace=True)
# df2.drop(columns='Unnamed: 0',inplace=True)
df2.head()                                       

In [ ]:
df2.to_excel('C:/Vivo Reports/5G/5G Mobilidade/TermPointToGNB_Distancia.xlsx', index=None)
print('TermPointToGNB_Distancia, Done')